# LoRA Trainer · Colab

UI-оболочка над kohya-ss/sd-scripts с поддержкой LoRA, LyCORIS и LoKr для NoobAI / Illustrious / Anima.

1. **Cell 1** — выберите GPU runtime и смонтируйте Drive.
2. **Cell 2** — укажите 4 пути (датасет, модели, output, samples).
3. **Cell 3** — установка зависимостей (~3 мин при первом запуске).
4. **Cell 4** — запуск сервера и cloudflared-туннеля.

В терминальном выводе появится `https://<random>.trycloudflare.com` — открывайте его в браузере.

## 1. GPU и Google Drive

In [ ]:
import subprocess, os
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv']).decode())

from google.colab import drive
drive.mount('/content/drive')

## 2. Папки

Отредактируйте 4 пути под себя. Все остальные настройки потом меняются прямо в UI.

**Структура датасета** (kohya-style): внутри `DATASET_ROOT` должны лежать подпапки вида `10_concept`, где `10` — это repeats.

In [ ]:
import os

# ===== ОТРЕДАКТИРУЙТЕ ЭТИ 4 ПУТИ =====
DATASET_ROOT    = '/content/drive/MyDrive/lora/datasets/my-concept'
BASE_MODEL_ROOT = '/content/drive/MyDrive/lora/models'
OUTPUT_ROOT     = '/content/drive/MyDrive/lora/output/my-concept'
SAMPLES_ROOT    = ''  # по умолчанию = OUTPUT_ROOT/sample
# ======================================

os.makedirs(OUTPUT_ROOT, exist_ok=True)
if SAMPLES_ROOT:
    os.makedirs(SAMPLES_ROOT, exist_ok=True)

for label, p in [('dataset', DATASET_ROOT), ('models', BASE_MODEL_ROOT), ('output', OUTPUT_ROOT)]:
    exists = os.path.isdir(p)
    print(f'{"OK" if exists else "MISSING"}  {label:8s}  {p}')

# expose to the FastAPI process
os.environ['LT_DATASET_ROOT']    = DATASET_ROOT
os.environ['LT_BASE_MODEL_ROOT'] = BASE_MODEL_ROOT
os.environ['LT_OUTPUT_ROOT']     = OUTPUT_ROOT
os.environ['LT_SAMPLES_ROOT']    = SAMPLES_ROOT
os.environ['LT_SD_SCRIPTS_DIR']  = '/content/sd-scripts'

## 3. Установка

In [ ]:
%cd /content

# --- kohya-ss/sd-scripts -------------------------------------------
![ -d /content/sd-scripts ] || git clone --depth=1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts

# --- chimerra-lora-trainer UI --------------------------------------
UI_REPO = 'https://github.com/DualChimerra/chimerra-lora-trainer.git'
![ -d /content/chimerra-lora-trainer ] || git clone --depth=1 $UI_REPO /content/chimerra-lora-trainer

# --- python deps ----------------------------------------------------
!pip install -q -r /content/sd-scripts/requirements.txt
!pip install -q lycoris-lora bitsandbytes prodigyopt fastapi 'uvicorn[standard]' python-multipart

# --- cloudflared ----------------------------------------------------
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

print('\n✓ Установка завершена.')

## 4. Запуск UI

Запускается FastAPI на :7860 и cloudflared-туннель. Ссылка появится ниже — открывайте её в браузере.

In [ ]:
import os, subprocess, time, re, signal, atexit, threading, sys

REPO_DIR = '/content/chimerra-lora-trainer'
PORT = 7860

# kill any previous instances
subprocess.run(['pkill', '-f', 'uvicorn'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.app:app', '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True,
)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True,
)

def _cleanup():
    for p in (server, tunnel):
        try: p.send_signal(signal.SIGINT); p.wait(timeout=5)
        except Exception:
            try: p.kill()
            except Exception: pass
atexit.register(_cleanup)

# --- wait for the trycloudflare URL ----------------------------------
tunnel_url = None
rx = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
deadline = time.time() + 60
while time.time() < deadline and tunnel_url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2); continue
    sys.stdout.write(line)
    m = rx.search(line)
    if m: tunnel_url = m.group(0)

if tunnel_url:
    print('\n' + '═'*64)
    print(f'  UI:  {tunnel_url}')
    print('═'*64 + '\n')
else:
    print('Не удалось получить URL туннеля. Посмотрите логи выше.')

# stream server logs into the cell output in the background
def _stream(p, prefix):
    for line in p.stdout:
        sys.stdout.write(f'[{prefix}] {line}')
threading.Thread(target=_stream, args=(server, 'srv'), daemon=True).start()
threading.Thread(target=_stream, args=(tunnel, 'cf' ), daemon=True).start()

# keep this cell alive so the processes don't die
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    _cleanup()